# 02 - Baselines B0 y B1

B0 predice la prevalencia de victorias en train. B1 usa una puntuación fija de recursos y aprende en train solo a traducirla a probabilidad. Ambos se ajustan por separado en A, B y C y se evalúan en val. Cada partida tiene el mismo peso total. No se usan calib ni test en este cuaderno.

Requiere haber corrido `00_arranque.ipynb` en el mismo servidor de Colab.

In [ ]:
# Preparar este cuaderno. 00_arranque debe haberse ejecutado en este runtime.
from pathlib import Path
import gc, sys
assert Path('/content/repo').exists(), 'Falta el repo: ejecuta 00_arranque.ipynb en este runtime'
assert list(Path('/content/datos').glob('estados_*.parquet')), 'Faltan datos: ejecuta 00_arranque.ipynb en este runtime'
%cd /content/repo
if '/content/repo/src' not in sys.path:
    sys.path.insert(0, '/content/repo/src')
DATA_DIR = Path('/content/datos')
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss
from ptcg_wp.data import load_scenario
SCENARIOS = ('A', 'B', 'C')
print('Entorno listo; tablas disponibles:', len(list(DATA_DIR.glob('*.parquet'))), '| escenarios:', SCENARIOS)

## Peso igual por partida

Una partida con 200 estados da peso `1/200` a cada estado; una con 20 da `1/20`. Cada partida suma 1 en total. Para una métrica por etapa, los pesos se recalculan dentro de esa etapa.

In [ ]:
def pesos_por_partida(game_ids):
    ids = pd.Series(game_ids, copy=False)
    return ids.map(ids.value_counts(sort=False)).rdiv(1.0).to_numpy(dtype=float)

def metricas(y, p, game_ids):
    w = pesos_por_partida(game_ids)
    y = np.asarray(y, dtype=int)
    p = np.clip(np.asarray(p, dtype=float), 1e-15, 1 - 1e-15)
    return {
        'auc': roc_auc_score(y, p, sample_weight=w),
        'log_loss': log_loss(y, p, sample_weight=w, labels=[0, 1]),
        'brier': brier_score_loss(y, p, sample_weight=w),
        'n_estados': len(y),
        'n_partidas': pd.Series(game_ids).nunique(),
    }

def evaluar(val, pred, escenario, modelo):
    filas = []
    grupos = [('global', np.ones(len(val), dtype=bool))]
    grupos += [(etapa, val['stage'].to_numpy() == etapa) for etapa in ('inicio', 'medio', 'final')]
    for etapa, mascara in grupos:
        parte = val.loc[mascara]
        if len(parte):
            filas.append({'escenario': escenario, 'modelo': modelo, 'etapa': etapa, **metricas(parte.y, pred[mascara], parte.game_id)})
    return filas
print('Funciones de ponderación y métricas listas.')

## B0 - probabilidad a priori

B0 ignora el tablero. Calcula en train la fracción de partidas ganadas por quien actúa y predice esa probabilidad en todos los estados de val. Es el piso de comparación.

## B1 - heurística de recursos

**Propuesta concreta para revisar:** `score = (opp_prizes - own_prizes) + diff_total_hp / hp_unit + diff_bench_n`. El signo queda positivo cuando el estado favorece al jugador que actúa. `hp_unit` será el promedio de `active_max_hp` positivo observado en los estados de `train`, ponderando para que cada partida tenga el mismo peso. Así, una ventaja en PV se expresa en unidades del HP promedio observado; no elegimos 100 sin evidencia. Este promedio se calcula por escenario y no usa `y` ni los IDs de carta.

Después, una regresión logística de una sola variable aprende en train la pendiente y el intercepto que convierten ese score en probabilidad. No aprende pesos separados para premios, PV y banca. Esta fórmula es una propuesta basada en recursos del juego; hay que revisarla antes de interpretar los resultados como evidencia final.

In [ ]:
def score_b1(df, hp_unit):
    ventaja_premios = df['opp_prizes'] - df['own_prizes']
    return (ventaja_premios + df['diff_total_hp'] / hp_unit + df['diff_bench_n']).to_numpy(dtype=float)

resultados, ajustes = [], []
columnas = ['game_id', 'y', 'stage', 'own_prizes', 'opp_prizes', 'own_active_max_hp', 'opp_active_max_hp', 'diff_total_hp', 'diff_bench_n']
columnas_parquet = [columna for columna in columnas if columna != 'stage']
for escenario in SCENARIOS:
    # Cargar un escenario a la vez para cuidar la RAM de Colab.
    states, feats, split, stage = load_scenario(DATA_DIR, escenario, columns=columnas_parquet)
    m_train, m_val = split == 'train', split == 'val'
    train = states.loc[m_train, columnas].copy()
    val = states.loc[m_val, columnas].copy()
    train['stage'], val['stage'] = stage[m_train], stage[m_val]
    w_train = pesos_por_partida(train.game_id)

    # B0: prevalencia de train, ponderada para dar el mismo peso a cada partida.
    prevalencia = np.average(train.y, weights=w_train)
    resultados += evaluar(val, np.full(len(val), prevalencia), escenario, 'B0')

    # Escala descriptiva de HP obtenida solo de train, sin etiquetas ni IDs de carta.
    hp_values = np.concatenate([train.own_active_max_hp.to_numpy(), train.opp_active_max_hp.to_numpy()])
    hp_weights = np.concatenate([w_train, w_train])
    hp_valid = hp_values > 0  # -1 indica que no hay Pokémon activo.
    hp_unit = np.average(hp_values[hp_valid], weights=hp_weights[hp_valid])

    # B1: score de dominio; solo se ajusta su conversión a probabilidad.
    s_train = score_b1(train, hp_unit).reshape(-1, 1)
    s_val = score_b1(val, hp_unit).reshape(-1, 1)
    calibrador = LogisticRegression(penalty=None, solver='lbfgs', max_iter=1000)
    calibrador.fit(s_train, train.y, sample_weight=w_train)
    p_b1 = calibrador.predict_proba(s_val)[:, 1]
    resultados += evaluar(val, p_b1, escenario, 'B1')
    ajustes.append({'escenario': escenario, 'prevalencia_B0_train': prevalencia, 'hp_unit_train': hp_unit,
                    'intercepto_B1': calibrador.intercept_[0], 'pendiente_B1': calibrador.coef_[0, 0],
                    'partidas_train': train.game_id.nunique(), 'partidas_val': val.game_id.nunique()})
    print(f'Escenario {escenario}: listo')
    del states, feats, split, stage, train, val, w_train, s_train, s_val, calibrador, hp_values, hp_weights
    gc.collect()

metricas_baselines = pd.DataFrame(resultados)
ajustes_baselines = pd.DataFrame(ajustes)
display(metricas_baselines.round(4))
display(ajustes_baselines.round(4))
metricas_baselines.to_csv('/content/metricas_B0_B1.csv', index=False)
ajustes_baselines.to_csv('/content/ajustes_B0_B1.csv', index=False)
print('Resúmenes guardados en /content/metricas_B0_B1.csv y /content/ajustes_B0_B1.csv')

## Leer los resultados

AUC más alto indica mejor ordenamiento de estados; log loss y Brier más bajos indican mejores probabilidades. `n_partidas` dice cuántas partidas contribuyeron. `pendiente_B1` e `intercepto_B1` ajustan la conversión del score a probabilidad; no son pesos aprendidos para las tres características.

La evaluación aquí es solo en val. No cambies la fórmula mirando test.